# Day 3 Homework — Recognize a Deliberate Error and Fix the Input

**Course:** LLM Application Engineering · SDA-AIE-213  
**Instructor:** Meaad Al-Marri  
**Assignment:** Extra homework, Day 3 (input-error practice)  
**Environment:** Python 3 · CPU · standard library only. No model, no API key, no package install.

A validator checks a public question **before** it would reach any model. Invalid input is rejected with `raise ValueError(...)`. The test code catches each rejection with `try / except` and records it as an **expected, successful** result, not as a crash.

## Step 1 — The validator

Two rules, in this order:
1. The input must be a string that still has text after removing spaces. Otherwise: `Enter a non-empty question string.`
2. The input must be at most 500 characters. Otherwise: `Use at most 500 characters.`

If both rules pass, the function returns the question with the outer spaces removed.

In [1]:
import json

MAX_CHARS = 500


def validate_public_question(question):
    # Rule 1: must be a real, non-blank string (None, numbers and '   ' all fail here)
    if not isinstance(question, str) or not question.strip():
        raise ValueError('Enter a non-empty question string.')
    # Rule 2: length limit
    if len(question) > MAX_CHARS:
        raise ValueError(f'Use at most {MAX_CHARS} characters.')
    return question.strip()

print('validator ready | max characters:', MAX_CHARS)

validator ready | max characters: 500


## Step 2 — Predict, then trigger the three deliberate errors

My prediction **before** running:

| Input | Predicted result | Why |
|---|---|---|
| `None` | `ValueError: Enter a non-empty question string.` | Not a string at all |
| `'   '` (blank spaces) | `ValueError: Enter a non-empty question string.` | A string, but empty after `.strip()` |
| `'x' * 501` | `ValueError: Use at most 500 characters.` | Passes rule 1, but is 1 character over the limit |

Each call sits inside `try`. If the validator raises, `except ValueError` catches it, and the test checks that the message is exactly the expected one. If an invalid input were **accepted**, the `else` branch would fail the test.

In [2]:
INVALID_CASES = [
    ('None',             None,      'Enter a non-empty question string.'),
    ('blank spaces',     '   ',     'Enter a non-empty question string.'),
    ('501 characters',   'x' * 501, 'Use at most 500 characters.'),
]

caught = []
for label, value, expected_message in INVALID_CASES:
    try:
        validate_public_question(value)
    except ValueError as error:
        assert str(error) == expected_message, (label, str(error))
        caught.append({'error_type': type(error).__name__,
                       'message': str(error),
                       'expected_rejection': True})
        print(f'{label:<16} -> caught {type(error).__name__}: {error}   (expected rejection ✔)')
    else:
        raise AssertionError(f'{label}: invalid input was accepted')

print(f'\n{len(caught)}/{len(INVALID_CASES)} deliberate errors caught with the exact expected message')

None             -> caught ValueError: Enter a non-empty question string.   (expected rejection ✔)
blank spaces     -> caught ValueError: Enter a non-empty question string.   (expected rejection ✔)
501 characters   -> caught ValueError: Use at most 500 characters.   (expected rejection ✔)

3/3 deliberate errors caught with the exact expected message


## Step 3 — Correct each input and show it is accepted

| Wrong input | My correction |
|---|---|
| `None` | Give an actual question string |
| `'   '` | Type real text instead of spaces |
| 501 characters | Shorten it to 500 characters or less |

I also test the **boundary**: exactly 500 characters must be accepted, and 501 must not. This shows the limit is `> 500` and not `>= 500`.

In [3]:
corrected = {
    'string instead of None':  'How can I check my application status?',
    'text instead of blanks':  '  How can I check my application status?  ',
    'shortened to 500 chars':  'x' * 500,
}

for label, value in corrected.items():
    accepted = validate_public_question(value)
    print(f'{label:<24} -> accepted ({len(accepted)} chars): {accepted[:45]!r}{"..." if len(accepted) > 45 else ""}')

# The official corrected input: outer spaces are removed
assert validate_public_question(' How can I check my application status? ') == 'How can I check my application status?'
# Boundary check: 500 passes, 501 fails
assert len(validate_public_question('x' * 500)) == 500
print('\nboundary check: 500 chars accepted, 501 chars rejected ✔')

string instead of None   -> accepted (38 chars): 'How can I check my application status?'
text instead of blanks   -> accepted (38 chars): 'How can I check my application status?'
shortened to 500 chars   -> accepted (500 chars): 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'...

boundary check: 500 chars accepted, 501 chars rejected ✔


## Step 4 — Save the report and compare it with the expected output

The report uses the same schema as the course's `extensions/expected/errors_result.json`.

In [4]:
import os

result = {
    'intentional_caught_errors': caught,
    'corrected_input': 'accepted',
    'checks': 'passed',
}

EXPECTED = {
    'intentional_caught_errors': [
        {'error_type': 'ValueError', 'message': 'Enter a non-empty question string.', 'expected_rejection': True},
        {'error_type': 'ValueError', 'message': 'Enter a non-empty question string.', 'expected_rejection': True},
        {'error_type': 'ValueError', 'message': 'Use at most 500 characters.', 'expected_rejection': True},
    ],
    'corrected_input': 'accepted',
    'checks': 'passed',
}
print('matches expected errors_result.json:', result == EXPECTED)

os.makedirs('murshid_outputs', exist_ok=True)
with open('murshid_outputs/day3_errors_result.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(json.dumps(result, indent=2))

matches expected errors_result.json: True
{
  "intentional_caught_errors": [
    {
      "error_type": "ValueError",
      "message": "Enter a non-empty question string.",
      "expected_rejection": true
    },
    {
      "error_type": "ValueError",
      "message": "Enter a non-empty question string.",
      "expected_rejection": true
    },
    {
      "error_type": "ValueError",
      "message": "Use at most 500 characters.",
      "expected_rejection": true
    }
  ],
  "corrected_input": "accepted",
  "checks": "passed"
}


## Reflection

### `raise`, `try` and `except`
- **`raise`** is how the validator *stops* and reports a problem. `raise ValueError('...')` ends the function right away and sends an error carrying a clear message. The bad input never moves on to the next step (for example, a model call).
- **`try`** marks code that *might* raise an error. Each validation call runs inside a `try` block.
- **`except ValueError as error`** *catches* that specific error type so the program keeps running instead of crashing. Inside it, I read `str(error)` to check the exact message and record the result.
- **`else`** (after `try/except`) runs only if **no** error happened. Here that means an invalid input slipped through, so the test fails on purpose.

### Why a caught rejection can be a successful test
The goal of this test is to prove that **bad input is blocked**. For `None`, blank text and 501 characters, the *correct* behaviour is to refuse. So when the `ValueError` is raised with the exact expected message and caught, the test **passed**. The real failure would be the opposite: the validator **accepting** invalid input, or raising a *different* error than expected.

This is different from a real runtime failure, like `ModuleNotFoundError`, a disconnected runtime, an unexpected exception, or a failed `assert`. Those mean something is actually broken.

### Limitations
- This is a standalone validation exercise. It does not change the functions in my core project notebook.
- Length is counted in Python characters, not model tokens.
- The validator only checks shape and length. It doesn't judge whether the question makes sense, and it doesn't detect sensitive data.